# Séance 3 · SQL et Git, les outils du quotidien · ⭐

**Niveau : ⭐ Débutant**

Deux outils que tous les développeurs et data scientists utilisent chaque jour : **SQL** pour interroger une base de données,
et **Git/GitHub** pour sauvegarder son travail en versions et le montrer. Ce notebook tourne dans **Google Colab** : rien à installer. `Maj + Entrée` pour exécuter.

**Livrable de la séance** : ton projet de la séance 2 est en ligne sur GitHub, avec un README qui l'explique.


## Préparation

On recharge le dataset Pokémon, on **renomme les colonnes en français** (plus simple à écrire en SQL), puis on le range dans une
base **SQLite** en mémoire. `sqlite3` fait partie de Python : rien à installer.

In [ ]:
import sqlite3
import pandas as pd

url = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    df = pd.read_csv(url)
    print("Dataset Pokémon chargé :", len(df), "lignes")
except Exception as e:
    print("Pas de réseau ? On utilise une mini-version de secours.", e)
    df = pd.DataFrame({
        "#": [1, 4, 7, 25, 94, 130, 143, 150], "Name": ["Bulbasaur", "Charmander", "Squirtle", "Pikachu", "Gengar", "Gyarados", "Snorlax", "Mewtwo"],
        "Type 1": ["Grass", "Fire", "Water", "Electric", "Ghost", "Water", "Normal", "Psychic"], "Type 2": ["Poison", None, None, None, "Poison", "Flying", None, None],
        "Total": [318, 309, 314, 320, 500, 540, 540, 680], "HP": [45, 39, 44, 35, 60, 95, 160, 106], "Attack": [49, 52, 48, 55, 65, 125, 110, 110],
        "Defense": [49, 43, 65, 40, 60, 79, 65, 90], "Sp. Atk": [65, 60, 50, 50, 130, 60, 65, 154], "Sp. Def": [65, 50, 64, 50, 75, 100, 110, 90],
        "Speed": [45, 65, 43, 90, 110, 81, 30, 130], "Generation": [1] * 8, "Legendary": [False] * 7 + [True],
    })

df = df.rename(columns={
    "#": "numero", "Name": "nom", "Type 1": "type1", "Type 2": "type2", "Total": "total", "HP": "pv", "Attack": "attaque",
    "Defense": "defense", "Sp. Atk": "attaque_spe", "Sp. Def": "defense_spe", "Speed": "vitesse", "Generation": "generation", "Legendary": "legendaire",
})
df["legendaire"] = df["legendaire"].astype(int)     # SQLite n'a pas de vrai/faux : 1 = légendaire, 0 = non
df.head(3)

In [ ]:
con = sqlite3.connect(":memory:")              # une base de données qui vit dans la mémoire de Colab
df.to_sql("pokemon", con, index=False)         # le tableau pandas devient une table SQL

def sql(requete):
    """Exécute une requête SQL et renvoie le résultat sous forme de tableau pandas."""
    return pd.read_sql_query(requete, con)

sql("SELECT * FROM pokemon LIMIT 3")

## 1. Où sont stockées les données d'une appli ?

Derrière Instagram, Spotify, ton jeu préféré ou Doctolib, il y a une **base de données** : un ensemble de tableaux géants
(les **tables**) rangés sur des serveurs. Une table = un onglet de tableur : des lignes (les utilisateurs, les messages, les scores) et des colonnes.

**SQL** (*Structured Query Language*) est la langue pour poser des questions à ces tables. Elle a 50 ans et tout le monde la parle encore : Instagram, la SNCF, ta banque.

**SQLite**, c'est une base de données complète qui tient dans un seul fichier. Il y en a déjà des dizaines dans ton téléphone (contacts, SMS, historique du navigateur).

In [ ]:
# Quelles tables y a-t-il dans notre base ? (sqlite_master est l'annuaire de la base)
sql("SELECT name, type FROM sqlite_master")

In [ ]:
# Quelles colonnes a la table pokemon ?
sql("PRAGMA table_info(pokemon)")[["name", "type"]]

## 2. SELECT et WHERE : choisir des colonnes et des lignes

Une requête SQL se lit presque comme une phrase : *SÉLECTIONNE ces colonnes DEPUIS cette table OÙ cette condition est vraie*.

- `SELECT nom, vitesse` : les colonnes (ou `*` pour toutes)
- `FROM pokemon` : la table
- `WHERE type1 = 'Fire'` : la condition (texte entre **guillemets simples**)
- `LIMIT 5` : seulement 5 lignes

In [ ]:
sql("""
SELECT nom, type1, vitesse
FROM pokemon
WHERE type1 = 'Fire'
LIMIT 5
""")

In [ ]:
# Plusieurs conditions : AND, OR. Compter : COUNT(*)
sql("""
SELECT COUNT(*) AS nb_rapides_et_forts
FROM pokemon
WHERE vitesse > 100 AND attaque > 100
""")

**Exercice** : affiche le nom et les PV des Pokémon de type Eau (`'Water'`) qui ont plus de 100 PV. Puis compte les Pokémon légendaires (`legendaire = 1`).

In [ ]:
# À toi
sql("SELECT nom, pv FROM pokemon LIMIT 3")

<details><summary>Solution</summary>

```python
print(sql("SELECT nom, pv FROM pokemon WHERE type1 = 'Water' AND pv > 100"))
print(sql("SELECT COUNT(*) AS nb_legendaires FROM pokemon WHERE legendaire = 1"))
```
</details>

## 3. ORDER BY : trier

`ORDER BY vitesse DESC` trie du plus grand au plus petit (`ASC` = croissant, c'est le défaut). Combiné avec `LIMIT`, ça donne un top 5.

In [ ]:
sql("""
SELECT nom, type1, vitesse
FROM pokemon
ORDER BY vitesse DESC
LIMIT 5
""")

**Exercice** : les 5 Pokémon avec la meilleure attaque, puis les 5 Pokémon les **plus lents** de la génération 1.

In [ ]:
# À toi
sql("SELECT nom, attaque FROM pokemon ORDER BY attaque LIMIT 5")

<details><summary>Solution</summary>

```python
print(sql("SELECT nom, attaque FROM pokemon ORDER BY attaque DESC LIMIT 5"))
print(sql("SELECT nom, vitesse FROM pokemon WHERE generation = 1 ORDER BY vitesse ASC LIMIT 5"))
```
</details>

## 4. GROUP BY : regrouper et calculer

Comme `groupby` en pandas : un tas par catégorie, puis un calcul par tas. Les calculs s'appellent des **agrégations** : `COUNT`, `AVG` (moyenne), `SUM`, `MAX`, `MIN`.
`AS` donne un nom au résultat.

In [ ]:
sql("""
SELECT type1, COUNT(*) AS nb, ROUND(AVG(total), 1) AS total_moyen
FROM pokemon
GROUP BY type1
ORDER BY total_moyen DESC
""")

**Exercice** : la vitesse moyenne par génération, puis le nombre de Pokémon légendaires **par génération** (indice : `WHERE legendaire = 1` avant le `GROUP BY`).

In [ ]:
# À toi
sql("SELECT generation, COUNT(*) AS nb FROM pokemon GROUP BY generation")

<details><summary>Solution</summary>

```python
print(sql("SELECT generation, ROUND(AVG(vitesse), 1) AS vitesse_moyenne FROM pokemon GROUP BY generation"))
print(sql("SELECT generation, COUNT(*) AS nb_legendaires FROM pokemon WHERE legendaire = 1 GROUP BY generation"))
```
</details>

## 5. Un premier JOIN : relier deux tables

Dans une vraie base, les informations sont réparties dans plusieurs tables (les utilisateurs d'un côté, leurs commandes de l'autre).
Le **JOIN** les recolle grâce à une colonne commune, comme deux pièces de puzzle.

On crée à la main une petite table `types` : chaque type, contre quoi il est fort, contre quoi il est faible. La colonne commune avec `pokemon` : le type.

In [ ]:
types = pd.DataFrame({
    "type": ["Fire", "Water", "Grass", "Electric", "Ground", "Rock", "Ice", "Psychic", "Fighting", "Flying"],
    "fort_contre": ["Grass", "Fire", "Water", "Water", "Electric", "Fire", "Grass", "Fighting", "Normal", "Grass"],
    "faible_contre": ["Water", "Electric", "Fire", "Ground", "Water", "Water", "Fire", "Bug", "Psychic", "Electric"],
})
types.to_sql("types", con, index=False)
sql("SELECT * FROM types")

In [ ]:
sql("""
SELECT p.nom, p.type1, t.fort_contre, t.faible_contre
FROM pokemon AS p
JOIN types AS t ON p.type1 = t.type
LIMIT 8
""")

`p` et `t` sont des surnoms (**alias**) pour écrire moins. Attention : un `JOIN` simple **ignore** les Pokémon dont le type n'est pas dans `types`
(Bug, Normal, Poison...). Pour les garder quand même, on écrit `LEFT JOIN`.

**Exercice** : combien de Pokémon sont faibles contre l'eau (`faible_contre = 'Water'`) ? Puis : par `faible_contre`, le nombre de Pokémon (GROUP BY sur la table jointe).

In [ ]:
# À toi
sql("""
SELECT COUNT(*) AS nb
FROM pokemon AS p
JOIN types AS t ON p.type1 = t.type
""")

<details><summary>Solution</summary>

```python
print(sql("""
SELECT COUNT(*) AS nb_faibles_contre_eau
FROM pokemon AS p JOIN types AS t ON p.type1 = t.type
WHERE t.faible_contre = 'Water'
"""))

print(sql("""
SELECT t.faible_contre, COUNT(*) AS nb
FROM pokemon AS p JOIN types AS t ON p.type1 = t.type
GROUP BY t.faible_contre
ORDER BY nb DESC
"""))
```
</details>

## 6. Git et GitHub : sauvegarder son travail en versions

Tu connais les **points de sauvegarde** dans un jeu vidéo : tu peux revenir en arrière si tu rates un boss. **Git** fait ça pour du code.
**GitHub** est le cloud où tu stockes ces sauvegardes et où tu peux les montrer (c'est ton portfolio, celui que regardent les écoles et les employeurs).

| Commande | Dans le jeu vidéo | Ce que ça fait |
|---|---|---|
| `git add fichier` | choisir ce qu'on met dans la sauvegarde | prépare les fichiers modifiés |
| `git commit -m "message"` | créer le point de sauvegarde | enregistre une version, avec un message qui dit **pourquoi** |
| `git push` | envoyer la sauvegarde sur le cloud | copie tes commits sur GitHub |
| `git pull` | récupérer la sauvegarde sur une autre console | rapatrie ce qui a changé sur GitHub |

Un bon message de commit dit l'intention : `Ajoute le graphique vitesse par génération`, pas `modif`.

**Depuis Colab, pas besoin de taper ces commandes** : Colab fait le `add + commit + push` pour toi en un clic. Voici les étapes précises.

### Pousser ton notebook de la séance 2 sur GitHub (étapes précises)

1. Sur https://github.com, ouvre ton dépôt `mon-portfolio-ia` (créé à la séance 0). Si tu ne l'as pas : bouton vert **New** → nom `mon-portfolio-ia` → coche *Add a README file* → **Create repository**.
2. Dans Colab, ouvre ton notebook `02_python_pour_la_data.ipynb` (menu **Fichier → Ouvrir un notebook**, onglet *Google Drive*).
3. Menu **Fichier → Enregistrer une copie sur GitHub**. La première fois, GitHub demande d'autoriser Colab : accepte.
4. Dans la fenêtre : *Dépôt* = `ton-pseudo/mon-portfolio-ia`, *Branche* = `main`, *Chemin du fichier* = `seance-02/02_python_pour_la_data.ipynb`,
   *Message de commit* = `Ajoute mon analyse Pokémon : 3 questions, 3 graphiques`. Coche *Inclure un lien vers Colab*. Clique **OK**.
5. Retourne sur GitHub, actualise : ton notebook est là, lisible avec ses graphiques. C'est ton premier commit.
6. Écris le README : sur la page du dépôt, clique sur le crayon ✏️ du `README.md`, colle le texte généré par la cellule ci-dessous, puis **Commit changes**.

Tu viens d'apprendre `add`, `commit` et `push` sans le savoir. `pull`, c'est quand tu rouvriras ton notebook depuis GitHub sur un autre ordinateur (**Fichier → Ouvrir un notebook → onglet GitHub**).

In [ ]:
# À toi : complète ta fiche, la cellule génère le README à coller sur GitHub
mon_projet = {
    "titre": "Analyse des Pokémon avec pandas",
    "auteur": "ton pseudo",
    "dataset": "800 Pokémon et leurs stats (Kaggle : abcsds/pokemon)",
    "questions": [
        "Les légendaires sont-ils vraiment plus forts ?",
        "Quelle génération a les Pokémon les plus rapides ?",
        "Les Pokémon rapides sont-ils fragiles ?",
    ],
    "ce_que_j_ai_appris": "filtrer, trier, regrouper avec pandas et tracer 3 types de graphiques",
    "outils": "Python, pandas, matplotlib, Google Colab",
}

readme = f"""# {mon_projet['titre']}

Par {mon_projet['auteur']} · Atelier Python, Data Science et IA générative

## Le dataset
{mon_projet['dataset']}

## Mes 3 questions
""" + "\n".join(f"{i}. {q}" for i, q in enumerate(mon_projet["questions"], start=1)) + f"""

## Ce que j'ai appris
{mon_projet['ce_que_j_ai_appris']}

## Outils
{mon_projet['outils']}

Notebook : `seance-02/02_python_pour_la_data.ipynb`
"""
print(readme)

**Exercice** : sur GitHub, ouvre l'onglet **Commits** de ton dépôt (ou l'horloge « N commits »). Tu vois l'historique : chaque ligne est un point de sauvegarde, avec son message et sa date.
Fais une deuxième modification du README (ajoute une ligne « À faire ensuite »), commit : tu as maintenant 2 versions, et tu peux revenir à la première.

## 7. Projet : mini-défi SQL vs pandas, 3 fois

Même question, deux langues. À chaque fois, on obtient le résultat en SQL **et** en pandas, et la fonction `compare` vérifie que les deux disent la même chose.
C'est le meilleur moyen de retenir les deux : `WHERE` ↔ filtre, `ORDER BY` ↔ `sort_values`, `GROUP BY` ↔ `groupby`.

In [ ]:
def compare(resultat_sql, resultat_pandas):
    """Compare les valeurs (pas les noms de colonnes) de deux tableaux, arrondies à 2 décimales."""
    def valeurs(tableau):
        tableau = pd.DataFrame(tableau).reset_index(drop=True)
        return [tuple(round(v, 2) if isinstance(v, float) else v for v in ligne) for ligne in tableau.values.tolist()]
    if valeurs(resultat_sql) == valeurs(resultat_pandas):
        print("✅ SQL et pandas sont d'accord !")
    else:
        print("❌ Pas le même résultat.\n--- SQL ---\n", resultat_sql, "\n--- pandas ---\n", resultat_pandas)

# Défi 1 (exemple résolu) : combien de Pokémon par génération ?
en_sql = sql("SELECT generation, COUNT(*) AS nb FROM pokemon GROUP BY generation ORDER BY generation")
en_pandas = df.groupby("generation").size().reset_index(name="nb")
compare(en_sql, en_pandas)

**Défi 2** : les 5 Pokémon les plus rapides (nom et vitesse), du plus rapide au moins rapide.
Piège : plusieurs Pokémon ont la même vitesse, et SQL et pandas ne rangent pas les ex æquo dans le même ordre. Départage-les par le nom, des deux côtés (`ORDER BY vitesse DESC, nom`).
Indice pandas : `sort_values(["vitesse", "nom"], ascending=[False, True])[["nom", "vitesse"]].head(5)`.

In [ ]:
# À toi
en_sql = sql("SELECT nom, vitesse FROM pokemon LIMIT 5")
en_pandas = df[["nom", "vitesse"]].head(3)
compare(en_sql, en_pandas)

<details><summary>Solution</summary>

```python
en_sql = sql("SELECT nom, vitesse FROM pokemon ORDER BY vitesse DESC, nom LIMIT 5")
en_pandas = df.sort_values(["vitesse", "nom"], ascending=[False, True])[["nom", "vitesse"]].head(5)
compare(en_sql, en_pandas)
```
</details>

**Défi 3** : l'attaque moyenne par type (`type1`), arrondie à 1 décimale, classée de la plus forte à la plus faible, top 5.
Indice pandas : `groupby("type1")["attaque"].mean().round(1).sort_values(ascending=False).head(5).reset_index()`.

In [ ]:
# À toi
en_sql = sql("SELECT type1, ROUND(AVG(attaque), 1) AS attaque_moyenne FROM pokemon GROUP BY type1 LIMIT 5")
en_pandas = df.groupby("type1")["attaque"].mean().round(1).head(5).reset_index()
compare(en_sql, en_pandas)

<details><summary>Solution</summary>

```python
en_sql = sql("""
SELECT type1, ROUND(AVG(attaque), 1) AS attaque_moyenne
FROM pokemon GROUP BY type1
ORDER BY attaque_moyenne DESC LIMIT 5
""")
en_pandas = df.groupby("type1")["attaque"].mean().round(1).sort_values(ascending=False).head(5).reset_index()
compare(en_sql, en_pandas)
```
</details>

**Défi bonus** : invente ta propre question (avec un JOIN sur `types` si tu es chaud), résous-la en SQL et en pandas
(indice pandas pour le JOIN : `df.merge(types, left_on="type1", right_on="type")`).

In [ ]:
# À toi : ta question
en_sql = sql("SELECT COUNT(*) AS nb FROM pokemon")
en_pandas = pd.DataFrame({"nb": [len(df)]})
compare(en_sql, en_pandas)

### Dernière étape du projet : tout sur GitHub

Enregistre aussi **ce notebook** sur GitHub (Fichier → Enregistrer une copie sur GitHub, chemin `seance-03/03_sql_et_git.ipynb`, message
`Ajoute mes requêtes SQL et le défi SQL vs pandas`). Ton portfolio a maintenant 2 notebooks et un README.

## À retenir

- Une appli stocke ses données dans une **base de données** faite de **tables** ; **SQL** est la langue pour les interroger.
- `SELECT colonnes FROM table WHERE condition ORDER BY colonne DESC LIMIT n` : 80 % des requêtes du quotidien.
- `GROUP BY` + `COUNT / AVG / SUM / MAX` : un calcul par catégorie. `JOIN ... ON` : relier deux tables par une colonne commune.
- SQL et pandas font la même chose : `WHERE` ↔ filtre, `ORDER BY` ↔ `sort_values`, `GROUP BY` ↔ `groupby`, `JOIN` ↔ `merge`.
- **Git** = des points de sauvegarde de ton code ; **GitHub** = le cloud qui les garde et les montre.
- `add` (choisir), `commit -m` (sauvegarder avec un message qui dit pourquoi), `push` (envoyer), `pull` (récupérer).
- Un README explique ton projet en 30 secondes à quelqu'un qui ne l'a jamais vu.

## Pour montrer aux autres

1. Montre ton dépôt GitHub : le notebook, le README, l'historique des commits.
2. Laquelle des deux « langues » préfères-tu, SQL ou pandas ? Pour quel type de question ?
3. Ta question bonus : qu'as-tu trouvé ?

Liens gratuits
- S'entraîner au SQL en jouant : https://sqlbolt.com (en anglais, interactif) · https://sql.sh (en français)
- Git expliqué visuellement : https://learngitbranching.js.org/?locale=fr_FR
- Le guide GitHub pour débuter : https://docs.github.com/fr/get-started/start-your-journey/hello-world